In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "movie_genre"
v_esquema = "movie_silver"
v_tabla = "movies_genres"
v_partition = "file_date"
v_merge_condition = "target.movie_id = source.movie_id and target.genre_id= source.genre_id"

In [0]:
#1. leemos el archivo JSON usando "DataFrameReader" de Spark

# Define el la estructura personName
movie_genre_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("genreId", IntegerType(), True)
])

movie_genre_df = spark.read\
    .schema(movie_genre_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/{v_archivo}.json")

display(movie_genre_df)


In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas

movie_genre_final_df = movie_genre_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("genreId", "genre_id")

movie_genre_final_df = add_ingestion_date(movie_genre_final_df)
movie_genre_final_df = add_env(movie_genre_final_df)
final_df = add_file_date (movie_genre_final_df)

display(final_df)

In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Paso 3 - Guardar datos en datalake en formato parket y particionado por Movie_id

#movie_genre_final_df.write.mode("overwrite").partitionBy("movie_id").format("delta").saveAsTable("movie_silver.movies_genres")
#movie_genre_final_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {movie_genre_final_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


In [0]:
dbutils.notebook.exit("El notebook 06. Ingestion File movie_genre, termino correctamente")